In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, upper
from pyspark.sql.types import StringType, StructField, StructType

from pipelines.stream_utils import (
    read_autoloader_stream,
    write_delta_table_stream,
)

In [0]:
stream_entity = "clicks" 

target_path = "/Volumes/main/silver/ecommerce/clicks"
target_schema_location = "/Volumes/main/silver/ecommerce/_schemas/clicks"
checkpoint_location = "/Volumes/main/silver/ecommerce/_checkpoints/clicks"

In [0]:
# Reprocess = reset state (checkpoint/schema) + drop target, then read all files again.
dbutils.widgets.dropdown("reprocess", "false", ["false", "true"])
reprocess = dbutils.widgets.get("reprocess").lower() == "true"

if reprocess:
    print("reprocess=true: clearing checkpoint/schema and target before re-running")
    spark.sql("DROP TABLE IF EXISTS main.silver.clicks")
    for p in [checkpoint_location, target_schema_location, target_path, source_schema_location]:
        try:
            dbutils.fs.rm(p, True)
            print(f"removed: {p}")
        except Exception as exc:
            print(f"skip remove {p}: {exc}")


In [0]:
source_stream_df = read_autoloader_stream(
    source_path=source_path,
    schema_location=source_schema_location,
    schema_evolution_mode="addNewColumns"
)

In [0]:
silver_stream_df = (
    source_stream_df
    .withColumn("occurred_at", to_timestamp(col("occurred_at")))
    .withColumn("event_date", to_date(col("occurred_at")))
)

In [0]:
silver_query = write_delta_table_stream(
    df=silver_stream_df,
    table_name="main.silver.clicks",
    checkpoint_location=checkpoint_location,
    query_name="silver_clicks_available_now",
    partition_by=["event_date"],
    merge_schema=True
)

silver_query.awaitTermination()

In [0]:
%sql
select *
from main.silver.clicks